# 🎓 Student Performance Prediction

This notebook explores the student performance dataset (`student-mat.csv`) and trains a machine learning model (`RandomForestRegressor`) to predict the final grade ($G_3$) based on key study and performance indicators.

## Table of Contents
1. [Setup and Libraries](#setup-and-libraries)
2. [Data Loading & Exploration](#data-loading--exploration)
3. [Feature Selection](#feature-selection)
4. [Model Training](#model-training)
5. [Model Evaluation](#model-evaluation)
6. [Feature Importance](#feature-importance)
7. [Model Serialization](#model-serialization)

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Set style for professional-looking plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries imported and plotting theme set successfully.")

## 1. Data Loading & Exploration

We load the student performance dataset (`student-mat.csv`), inspect its structure, and generate basic summaries.

In [ ]:
# Load dataset using pathlib for robust path handling
BASE_DIR = Path(".").resolve()
csv_path = BASE_DIR / "data" / "student-mat.csv"

df = pd.read_csv(csv_path, sep=";")
print(f"Dataset loaded successfully. Shape: {df.shape}")
df.head()

### Summary Statistics
Let's inspect columns, data types, and check for any missing values.

In [ ]:
df.info()

In [ ]:
df.describe()

### Data Visualization
Let's visualize the target column, the final grade ($G_3$), to see its distribution.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['G3'], kde=True, color='teal', bins=20)
plt.title('Distribution of Final Grades (G3)', fontsize=14)
plt.xlabel('Final Grade (G3)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.show()

### Correlation matrix of key features
Let's look at the correlation coefficient between the chosen predictor variables (`studytime`, `failures`, `absences`, `G1`, `G2`) and the target `G3`.

In [ ]:
selected_features = ["studytime", "failures", "absences", "G1", "G2", "G3"]
corr_matrix = df[selected_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Selected Features", fontsize=14)
plt.show()

## 2. Feature Selection & Splits

We extract features ($X$) and target ($y$), and split them into training (80%) and testing (20%) datasets.

In [ ]:
# Define features and target
X = df[["studytime", "failures", "absences", "G1", "G2"]]
y = df["G3"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

## 3. Model Training

We fit a Random Forest Regressor on the training set.

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Random Forest Regressor trained successfully!")

## 4. Model Evaluation

Let's assess the model performance using $R^2$, MAE, and RMSE metrics.

In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("=" * 50)
print("Model Evaluation Performance")
print("=" * 50)
print(f"Accuracy (R² Score): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print("=" * 50)

### Actual vs. Predicted Visualization

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='darkblue', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs. Predicted Final Grades (G3)', fontsize=14)
plt.xlabel('Actual Grade', fontsize=12)
plt.ylabel('Predicted Grade', fontsize=12)
plt.show()

## 5. Feature Importance

Let's see which features contribute the most to predicting the final grade.

In [ ]:
importances = model.feature_importances_
features_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x='Importance', y='Feature', data=features_df, palette='viridis', hue='Feature', legend=False)
plt.title('Feature Importances in Student Performance Prediction', fontsize=14)
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.show()

## 6. Model Serialization

We save the model structure for inference in other parts of the project.

In [ ]:
model_path = BASE_DIR / "model.pkl"
with open(model_path, "wb") as file:
    pickle.dump(model, file)
print(f"✅ Trained model successfully saved to: {model_path}")